# Time Series Forecasting — Bike Sharing

Bike-sharing makes it easy to pick up a bicycle at one station and return it at another. For operators, anticipating when bikes will be in demand helps keep the service available where people need it. Hourly rental records also show how demand shifts with time and weather.

This experiment uses hourly records from Washington, D.C.’s Capital Bikeshare system in the UCI Bike Sharing dataset, covering 2011 and 2012. Along with calendar and weather details, each row records `cnt`—the number of rentals that hour—which is the value we aim to predict. [UCI Bike Sharing dataset](https://doi.org/10.24432/C5W894)

We compare a previous-week baseline with LightGBM and TabPFN using engineered rental lags, and AutoGluon Time Series using the rental history directly with previous-hour weather covariates. All models are evaluated on the same held-out test hours using MAE and RMSE.


## Installation

Install the libraries needed for data preparation, analysis, model training, and experiment tracking.


In [ ]:
import sys
!uv pip install --python "{sys.executable}" -q ucimlrepo fg-data-profiling scikit-learn mlflow optuna matplotlib shap lightgbm python-dotenv tabpfn 'tabpfn-extensions[interpretability]' 'autogluon.tabular[all]' 'autogluon.timeseries'
%matplotlib inline

## Dataset Loading

Load the UCI Bike Sharing dataset (ID 275), including its hourly features, rental-count target, and metadata. The `casual` and `registered` counts add up to `cnt` and are excluded from the feature set.


In [ ]:
from ucimlrepo import fetch_ucirepo

# Load the hourly bike-sharing records.
bike_sharing = fetch_ucirepo(id=275)

X = bike_sharing.data.features.copy()
y = bike_sharing.data.targets["cnt"].copy()

print(bike_sharing.metadata)
print(bike_sharing.variables)


## Dataset Overview

Inspect the dataset dimensions, sample records, feature types, and rental counts.


In [ ]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 rows:")
print(X.head())

print("\nColumn information:")
X.info()

print("\nRental count statistics:")
print(y.describe())

print("\nFeature statistics:")
display(X.describe(include="all").T)


## Exploratory Data Analysis (EDA)

Generate a profiling report to explore rental counts, feature distributions, and relationships in the dataset.


In [ ]:
import pandas as pd
from data_profiling import ProfileReport

df = pd.concat([X, y], axis=1)

# Generate the EDA report.
profile = ProfileReport(
    df,
    title="Bike Sharing Dataset Analysis",
    explorative=True,
)
profile.to_file("bike_sharing_report.html")

## Data Preparation

To forecast the next hour’s demand, we first need a reliable hourly timeline. The records have gaps, so we add the missing hours while keeping their unknown rental counts empty.

We can then give the models a view of recent demand through earlier rental counts and the previous hour’s weather. Once those features are in place, we keep the hours with known rental counts for training and evaluation.


In [ ]:
import pandas as pd

# 1. Combine features and target, then create and sort the hourly timestamp.
hourly_data = pd.concat([X, y], axis=1)
hourly_data["timestamp"] = pd.to_datetime(hourly_data["dteday"]) + pd.to_timedelta(hourly_data["hr"], unit="h")
hourly_data = hourly_data.sort_values("timestamp").set_index("timestamp")

# 2. Check for duplicate timestamps.
if hourly_data.index.has_duplicates:
    raise ValueError("Duplicate timestamps found; inspect them before continuing.")

# 3. Add missing hours and leave unknown values empty.
hours = pd.date_range(hourly_data.index.min(), hourly_data.index.max(), freq="h", name="timestamp")
missing_hours = len(hours) - len(hourly_data)
hourly_data = hourly_data.reindex(hours)

# 4. Add rental counts from earlier hours as lag features.
hourly_data["rentals_lag_1h"] = hourly_data["cnt"].shift(1)
hourly_data["rentals_lag_24h"] = hourly_data["cnt"].shift(24)
hourly_data["rentals_lag_168h"] = hourly_data["cnt"].shift(168)

# 5. Use the previous hour's weather for the next-hour forecast.
weather_columns = ["weathersit", "temp", "atemp", "hum", "windspeed"]
previous_weather = hourly_data[weather_columns].shift(1).add_suffix("_lag_1h")
hourly_data = pd.concat([hourly_data, previous_weather], axis=1)

# 6. Drop rows with missing targets and separate features from the target.
model_data = hourly_data.dropna(subset=["cnt"])
X = model_data.drop(columns=["dteday", "cnt"] + weather_columns)
y = model_data["cnt"].copy()
print(f"Added missing hours: {missing_hours}")
print(f"Known rental counts: {y.notna().sum():,}")


- The records have **165 missing hours**. We added them temporarily so each lag refers to the correct clock hour.
- Rental lags capture demand **one hour, one day, and one week earlier**. Weather features use the **previous hour** so they are available for the next-hour forecast.
- If an earlier hour is missing, its lag values stay empty. We then remove rows without a **known rental count**.


## Feature Type Conversion

Season and weather conditions are recorded as codes, so we mark them as categorical features. The remaining calendar fields and lag values stay numeric.


In [ ]:
categorical_cols = ["season", "weathersit_lag_1h"]
X[categorical_cols] = X[categorical_cols].astype("category")

print(X.dtypes)

## Development and Test Split

Reserve the latest 20% of recorded hours for testing and use the earlier records for development. Within development, three time-ordered folds are used to tune LightGBM.

This evaluates rolling one-hour-ahead forecasts, where earlier rental counts and weather observations become available before each prediction.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Keep the latest recorded hours separate for final testing.
split_idx = int(len(X) * 0.8)
X_dev, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_dev, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Reuse these time-ordered development folds for model tuning.
cv = TimeSeriesSplit(n_splits=3)
cv_splits = list(cv.split(X_dev))

print(f"Development: {len(X_dev):,} rows, {X_dev.index.min()} to {X_dev.index.max()}")
print(f"Test: {len(X_test):,} rows, {X_test.index.min()} to {X_test.index.max()}")
print(f"Development folds: {len(cv_splits)}")


## 1. LightGBM

### Hyperparameter Tuning

LightGBM can learn from the calendar, weather, and rental history features, including missing lag values. We use Optuna to find settings that minimize mean absolute error (MAE) across the three time-ordered development folds. MAE measures the average forecast error in rentals per hour.

Each trial and its fold scores are recorded in MLflow.


In [ ]:
import numpy as np
import mlflow
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

SEED = 42
N_TRIALS = 25
LIGHTGBM_DEVICE = "gpu"  # Use "cpu" if an OpenCL GPU is unavailable.
mlflow.set_experiment("Bike Sharing Forecast")

def train_and_score_cv(params):
    scores = []
    for train_idx, valid_idx in cv_splits:
        X_train, X_valid = X_dev.iloc[train_idx], X_dev.iloc[valid_idx]
        y_train, y_valid = y_dev.iloc[train_idx], y_dev.iloc[valid_idx]
        model = LGBMRegressor(
            **params, n_estimators=500, objective="regression_l1",
            device_type=LIGHTGBM_DEVICE, random_state=SEED, verbosity=-1,
        )
        model.fit(X_train, y_train)
        scores.append(mean_absolute_error(y_valid, model.predict(X_valid)))
    return scores

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 8, 64),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.001, 10.0, log=True),
    }
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True) as trial_run:
        mlflow.log_params(params)
        scores = train_and_score_cv(params)
        mlflow.log_metrics({f"fold_{i}_mae": score for i, score in enumerate(scores, 1)})
        mlflow.log_metric("mean_mae", float(np.mean(scores)))
        trial.set_user_attr("run_id", trial_run.info.run_id)
    return float(np.mean(scores))


### Model Selection

We first measure a simple baseline that predicts the rental count from the same hour one week earlier. Optuna then evaluates LightGBM settings on the same development folds and selects the trial with the lowest average MAE.

The baseline is scored where the previous-week count is available; LightGBM is scored on every validation row.


In [ ]:
with mlflow.start_run(run_name="bike_sharing_study") as run:
    mlflow.log_params({"n_estimators": 500, "objective": "regression_l1", "device_type": LIGHTGBM_DEVICE, "n_trials": N_TRIALS, "cv_folds": len(cv_splits)})

    # Score the previous-week baseline on available validation rows.
    baseline_scores = []
    for _, valid_idx in cv_splits:
        previous_week = X_dev.iloc[valid_idx]["rentals_lag_168h"]
        available = previous_week.notna()
        baseline_scores.append(mean_absolute_error(y_dev.iloc[valid_idx][available], previous_week[available]))
    mlflow.log_metric("baseline_cv_mae", float(np.mean(baseline_scores)))

    # Select the settings with the lowest mean validation MAE.
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_param("best_child_run_id", study.best_trial.user_attrs["run_id"])
    mlflow.log_metric("best_cv_mae", study.best_value)

    print(f"Previous-week baseline CV MAE: {np.mean(baseline_scores):.2f}")
    print(f"Best LightGBM CV MAE: {study.best_value:.2f}")


**Note:**

- LightGBM’s three `TimeSeriesSplit` folds use expanding training windows: each fold trains on all earlier development hours and validates on the next block.


### Test Evaluation

Fit LightGBM on all development records using the selected settings, then evaluate it on the later test period. MAE shows the average error in rentals per hour; RMSE gives more weight to large errors.

The baseline uses the count from the same hour one week earlier. Its scores are compared with LightGBM on the same hours where that count is available.


In [ ]:
from IPython.display import display
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_id=run.info.run_id):
    # 1. Fit the selected model on all development data.
    model = LGBMRegressor(
        **study.best_params, n_estimators=500, objective="regression_l1",
        device_type=LIGHTGBM_DEVICE, random_state=SEED, verbosity=-1,
    )
    model.fit(X_dev, y_dev)

    # 2. Evaluate the model and baseline on the test period.
    predictions = pd.Series(model.predict(X_test), index=X_test.index, name="predicted")
    previous_week = X_test["rentals_lag_168h"]
    available = previous_week.notna()

    def regression_scores(actual, predicted):
        return {
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        }

    model_test = regression_scores(y_test, predictions)
    model_matched = regression_scores(y_test[available], predictions[available])
    baseline_test = regression_scores(y_test[available], previous_week[available])
    test_results = pd.DataFrame({
        "LightGBM (all test hours)": model_test,
        "LightGBM (baseline hours)": model_matched,
        "Previous-week baseline": baseline_test,
    }).T
    display(test_results.round(2))
    print(f"Baseline available for {available.sum():,} of {len(y_test):,} test hours.")

    # 3. Record the metrics and fitted model.
    mlflow.log_metrics({f"test_model_{k.lower()}": v for k, v in model_test.items()})
    mlflow.log_metrics({f"test_model_matched_{k.lower()}": v for k, v in model_matched.items()})
    mlflow.log_metrics({f"test_baseline_{k.lower()}": v for k, v in baseline_test.items()})
    mlflow.log_metric("test_baseline_hours", int(available.sum()))
    mlflow.log_dict(test_results.to_dict(orient="index"), "test_metrics.json")
    mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

print(f"MLflow run ID: {run.info.run_id}")


### Forecast Plots

The first plot compares predictions with actual demand over one week of the test period. The second shows each test error, calculated as actual minus predicted rentals, to help spot patterns in overprediction and underprediction.


In [ ]:
import matplotlib.pyplot as plt

# Compare forecasts with actual demand over the first test week.
first_week = y_test.index < y_test.index.min() + pd.Timedelta(days=7)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test.index[first_week], y_test[first_week], label="Actual")
ax.plot(predictions.index[first_week], predictions[first_week], label="Predicted", alpha=0.8)
ax.set(title="Bike rentals during the first test week", xlabel="Time", ylabel="Rentals per hour")
ax.legend()
fig.tight_layout()
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_figure(fig, "plots/actual_vs_predicted.png")
display(fig)
plt.close(fig)



**Note:** Each point represents an hour. The model predicts one hour ahead using actual earlier rental counts and weather as they become available; it is not retrained for each prediction.


In [ ]:
# Inspect errors across the test period.
residuals = y_test - predictions
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(predictions, residuals, alpha=0.3, s=12)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set(title="Test forecast errors", xlabel="Predicted rentals", ylabel="Actual − predicted rentals")
fig.tight_layout()
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_figure(fig, "plots/residuals.png")
display(fig)
plt.close(fig)


**Note:** Each point represents a test hour. The error is actual minus predicted rentals. Points above zero are underpredictions, points below zero are overpredictions, and points near zero are more accurate.


#### Observations

- LightGBM’s test MAE is **29.40 rentals per hour**, close to its development MAE of **31.17**. Its test RMSE is **47.93**, reflecting some larger errors.
- On hours where the previous-week count is available, LightGBM’s MAE is **29.29**, compared with **64.48** for the baseline.
- During the first test week, predictions follow the daily rises and falls in demand and capture most peaks.
- Errors spread out more at higher predicted demand, with some substantial overpredictions and underpredictions.


### Model Explanations

SHAP estimates how each feature contributes to a forecast, both across a sample and for one hour. Contributions are measured in **rentals per hour**; they describe the fitted model, not causes. Use the held-out test sample only after model choices are complete.

In [ ]:
import shap

def explain_demand(model, X, sample_size=500):
    # Use a reproducible sample so plots match the classification experiment.
    sample = X.sample(n=min(sample_size, len(X)), random_state=SEED)
    values = shap.TreeExplainer(model)(sample)
    return sample, values

def log_shap_plot(fig, dataset, name):
    fig.tight_layout()
    with mlflow.start_run(run_id=run.info.run_id):
        mlflow.log_figure(fig, f"shap/{dataset}/{name}.png")
    display(fig)
    plt.close(fig)

def plot_shap_importance(explanation, dataset):
    shap.plots.bar(explanation, max_display=15, show=False)
    log_shap_plot(plt.gcf(), dataset, "feature_importance")

def plot_shap_beeswarm(explanation, dataset):
    shap.plots.beeswarm(explanation, max_display=15, show=False)
    log_shap_plot(plt.gcf(), dataset, "beeswarm")

def plot_shap_waterfall(explanation, dataset, row=0):
    shap.plots.waterfall(explanation[row], max_display=15, show=False)
    log_shap_plot(plt.gcf(), dataset, f"waterfall_{row}")

def plot_shap_scatter(explanation, dataset, feature):
    shap.plots.scatter(explanation[:, feature], show=False)
    log_shap_plot(plt.gcf(), dataset, f"scatter_{feature}")

# Switch to development only for exploratory checks; test explanations follow model selection.
SHAP_DATASET = "test"  # "development" or "test"
shap_data = {"development": X_dev, "test": X_test}[SHAP_DATASET]
shap_target = {"development": y_dev, "test": y_test}[SHAP_DATASET]
shap_sample, demand_shap = explain_demand(model, shap_data)
example_hour = shap_sample.index[0]
print(f"Example hour: {example_hour}")
print(f"Actual rentals: {shap_target.loc[example_hour]:.0f}")
print(f"Predicted rentals: {model.predict(shap_sample.iloc[[0]])[0]:.1f}")

#### SHAP Feature Importance

Rank features by their mean absolute contribution across the sample. Larger bars mean a feature moved forecasts more, but do not show the direction.

In [ ]:
plot_shap_importance(demand_shap, SHAP_DATASET)

#### SHAP Beeswarm

Show how each feature’s contribution varies across sampled hours. Points to the right raise the forecast; points to the left lower it. Colours show feature values, including the recorded codes for categorical features.

In [ ]:
plot_shap_beeswarm(demand_shap, SHAP_DATASET)

#### SHAP Waterfall

Explain the sampled hour printed above. Each feature moves the model’s baseline forecast up or down to its predicted rental count.

In [ ]:
plot_shap_waterfall(demand_shap, SHAP_DATASET, row=0)

#### SHAP Scatter Plot

Compare the previous hour’s rental count with its contribution to the next-hour forecast. Each point is a sampled hour; the vertical spread shows that the same lag value can contribute differently in different contexts.

In [ ]:
plot_shap_scatter(demand_shap, SHAP_DATASET, feature="rentals_lag_1h")

#### Observations

- **Previous-hour rentals** have the largest average influence in the sampled test hours, followed by previous-week rentals, hour of day, and previous-day rentals.
- Higher previous-hour rental counts generally push the forecast up, while lower counts push it down. The scatter shows this rising, curved pattern.
- For **2012-11-03 05:00**, the model predicted 8.5 rentals and the actual count was 9. Low recent rental lags pulled the forecast down from the baseline of about 174 rentals.
- These plots describe the model’s predictions. Related rental lags can share influence, and the patterns do not show that a feature causes demand.

## 2. TabPFN 3.5 Regression

TabPFN 3.5 is a pretrained model for tabular regression. Fit it on the development period and evaluate its rental-count forecasts on the later test period, using the existing lag and calendar features. TabPFN uses these columns as tabular inputs; it does not receive the timeline as a sequence.

On first use, weights download from [Prior Labs on Hugging Face](https://huggingface.co/Prior-Labs/tabpfn_3_5); follow the sign-in and license prompts. The weights are for non-commercial, non-production use. This test evaluation is exploratory because the test period has already been reviewed in the LightGBM experiment.

### Test Evaluation

In [ ]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

with mlflow.start_run(run_name="TabPFN 3.5 Regression") as tabpfn_run:
    # Fit on the development period and predict the later test period.
    tabpfn_model = TabPFNRegressor.create_default_for_version(
        ModelVersion.V3_5, random_state=SEED,
        fit_mode="fit_with_cache", keep_cache_on_device=False,
    )
    tabpfn_model.fit(X_dev, y_dev)

    # Predict in batches to reduce peak GPU memory during test inference.
    TABPFN_PREDICTION_BATCH_SIZE = 100
    prediction_batches = [
        tabpfn_model.predict(
            X_test.iloc[start:start + TABPFN_PREDICTION_BATCH_SIZE],
            output_type="mean",
        )
        for start in range(0, len(X_test), TABPFN_PREDICTION_BATCH_SIZE)
    ]
    tabpfn_predictions = pd.Series(
        np.concatenate(prediction_batches),
        index=X_test.index,
        name="TabPFN 3.5",
    )

    tabpfn_scores = regression_scores(y_test, tabpfn_predictions)
    display(pd.DataFrame([tabpfn_scores], index=["TabPFN 3.5"]).round(2))
    mlflow.log_params({
        "model_type": "TabPFNRegressor",
        "model_version": "V3_5",
        "fit_mode": "fit_with_cache",
        "keep_cache_on_device": False,
        "training_rows": len(X_dev),
        "test_rows": len(X_test),
        "prediction_batch_size": TABPFN_PREDICTION_BATCH_SIZE,
    })
    mlflow.log_metrics({f"test_{name.lower()}": value for name, value in tabpfn_scores.items()})

### Forecast Plot

In [ ]:
# Compare TabPFN forecasts with actual rentals over the first test week.
tabpfn_first_week = y_test.index < y_test.index.min() + pd.Timedelta(days=7)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test.index[tabpfn_first_week], y_test[tabpfn_first_week], label="Actual")
ax.plot(
    tabpfn_predictions.index[tabpfn_first_week],
    tabpfn_predictions[tabpfn_first_week],
    label="TabPFN 3.5",
    alpha=0.8,
)
ax.set(title="TabPFN 3.5 forecasts during the first test week", xlabel="Time", ylabel="Rentals per hour")
ax.legend()
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "plots/tabpfn_actual_vs_predicted.png")
display(fig)
plt.close(fig)

**Note:** Each point on the lines is one test hour. TabPFN predicts that hour’s rental count from the lag and calendar features available for it.

### Residual Plot

In [ ]:
# Inspect TabPFN errors across the test period.
tabpfn_residuals = y_test - tabpfn_predictions
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(tabpfn_predictions, tabpfn_residuals, alpha=0.3, s=12)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set(title="TabPFN 3.5 test forecast errors", xlabel="Predicted rentals", ylabel="Actual − predicted rentals")
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "plots/tabpfn_residuals.png")
display(fig)
plt.close(fig)

**Note:** Each point is a test hour. The error is actual minus predicted rentals. Points above zero are underpredictions; points below zero are overpredictions.

### Observations

- TabPFN follows the daily demand pattern and captures most of the large peaks in the first test week, though some sharp peaks are over- or underpredicted.
- Residuals sit mostly around zero at lower demand and spread out at higher predictions, where the larger forecast errors occur.

### Model Explanations

TabPFN is not a tree model, so its explanations use the Shapley method from `tabpfn-extensions`, then display the results with SHAP. To keep the repeated predictions manageable, explain a reproducible sample of 50 test hours with a limited evaluation budget. Contributions are approximate and measured in rentals per hour; they describe model behaviour, not causes.

In [ ]:
from tabpfn_extensions.interpretability import shapiq_to_shap_explanation
from tabpfn_extensions.interpretability.shapiq import get_tabpfn_imputation_explainer

TABPFN_SHAP_SAMPLE_SIZE = 50
TABPFN_SHAP_BUDGET = 64
tabpfn_shap_sample = X_test.sample(
    n=min(TABPFN_SHAP_SAMPLE_SIZE, len(X_test)), random_state=SEED
)
tabpfn_shap_reference = X_dev.sample(n=1, random_state=SEED)

# Restore pandas feature types when the explainer passes rows to TabPFN.
class TabPFNRegressionModel:
    fit_mode = tabpfn_model.fit_mode

    def predict(self, values):
        frame = pd.DataFrame(values, columns=X_dev.columns).astype(X_dev.dtypes.to_dict())
        return tabpfn_model.predict(frame, output_type="mean")

tabpfn_explainer = get_tabpfn_imputation_explainer(
    model=TabPFNRegressionModel(), data=tabpfn_shap_reference,
    index="SV", max_order=1, random_state=SEED,
)
tabpfn_shap = shapiq_to_shap_explanation(
    tabpfn_explainer, tabpfn_shap_sample, budget=TABPFN_SHAP_BUDGET,
    feature_names=list(X_dev.columns),
)

# Use the reference row's forecast as the baseline and check reconstruction.
reference_prediction = TabPFNRegressionModel().predict(tabpfn_shap_reference)[0]
tabpfn_shap.base_values = np.full(len(tabpfn_shap_sample), reference_prediction)
explained_predictions = tabpfn_predictions.loc[tabpfn_shap_sample.index].to_numpy()
reconstructed_predictions = tabpfn_shap.base_values + tabpfn_shap.values.sum(axis=1)
print(f"Reference forecast: {reference_prediction:.1f}")
print(
    "Largest SHAP reconstruction difference: "
    f"{np.max(np.abs(reconstructed_predictions - explained_predictions)):.2f} rentals"
)

example_hour = tabpfn_shap_sample.index[0]
print(f"Example hour: {example_hour}")
print(f"Actual rentals: {y_test.loc[example_hour]:.0f}")
print(f"Predicted rentals: {tabpfn_predictions.loc[example_hour]:.1f}")

#### SHAP Feature Importance

Rank features by their mean absolute contribution across the sampled hours. Larger bars mean a feature moved the forecasts more, but do not show the direction.

In [ ]:
shap.plots.bar(tabpfn_shap, max_display=15, show=False)
fig = plt.gcf()
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "shap/test/tabpfn_3_5/feature_importance.png")
display(fig)
plt.close(fig)

#### SHAP Beeswarm

Show each feature’s estimated contribution across sampled hours. Points to the right raise the forecast; points to the left lower it. Colours show feature values, including the recorded codes for categorical features.

In [ ]:
shap.plots.beeswarm(tabpfn_shap, max_display=15, show=False)
fig = plt.gcf()
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "shap/test/tabpfn_3_5/beeswarm.png")
display(fig)
plt.close(fig)

#### SHAP Waterfall

Explain the sampled hour printed above. The contributions move the explainer’s reference forecast up or down to TabPFN’s prediction.

In [ ]:
shap.plots.waterfall(tabpfn_shap[0], max_display=15, show=False)
fig = plt.gcf()
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "shap/test/tabpfn_3_5/waterfall_0.png")
display(fig)
plt.close(fig)

**Note:** The waterfall starts from the development reference forecast. The printed reconstruction difference checks how closely the estimated contributions add up to TabPFN’s forecast.

#### SHAP Scatter Plot

Compare the previous hour’s rental count with its estimated contribution to the next-hour forecast. Each point is a sampled test hour; the vertical axis is in rentals per hour.

In [ ]:
shap.plots.scatter(tabpfn_shap[:, "rentals_lag_1h"], show=False)
fig = plt.gcf()
fig.tight_layout()
with mlflow.start_run(run_id=tabpfn_run.info.run_id):
    mlflow.log_figure(fig, "shap/test/tabpfn_3_5/scatter_rentals_lag_1h.png")
display(fig)
plt.close(fig)

#### Observations

- **Previous-hour rentals** rank first for both models. TabPFN gives **hour of day** the second-largest average influence, while LightGBM ranks the **previous-week rental count** second.
- In both beeswarm and lag scatter plots, higher previous-hour rental counts generally push forecasts up; TabPFN’s relationship is nonlinear.
- TabPFN SHAP is approximate and uses 50 test hours with a limited budget; LightGBM TreeSHAP uses 500. Compare the main features and directions, not the bar magnitudes.

## 3. AutoGluon Time Series Regression

AutoGluon’s `TimeSeriesPredictor` is designed for forecasting. It uses the `cnt` history directly and the previous-hour weather as known covariates. The best-quality preset evaluates models on multiple chronological windows with expanding training history, then ensembles their forecasts. Each validation window predicts one hour ahead. After model selection on development data, we evaluate one-hour-ahead forecasts across the held-out test period.


### Development Evaluation

Keep the complete hourly timeline through the end of development, including hours with unknown rental counts. AutoGluon uses these as missing targets while fitting and validates on later time windows.


In [ ]:
from importlib.metadata import version
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

AG_TS_TARGET = "cnt"
AG_TS_ITEM_ID = "bike_sharing"
AG_TS_KNOWN_COVARIATES = [f"{column}_lag_1h" for column in weather_columns]

# Keep the complete hourly grid, but stop before the held-out test period.
ag_ts_dev_frame = hourly_data.loc[
    :X_dev.index.max(), [AG_TS_TARGET] + AG_TS_KNOWN_COVARIATES
].copy()
ag_ts_dev_frame["weathersit_lag_1h"] = ag_ts_dev_frame["weathersit_lag_1h"].astype("category")
ag_ts_dev_frame["item_id"] = AG_TS_ITEM_ID
ag_ts_dev_frame = ag_ts_dev_frame.rename_axis("timestamp").reset_index()
ag_ts_dev_data = TimeSeriesDataFrame.from_data_frame(
    ag_ts_dev_frame,
    id_column="item_id",
    timestamp_column="timestamp",
)

with mlflow.start_run(run_name="autogluon_timeseries_bike_sharing") as autogluon_ts_run:
    autogluon_ts_predictor = TimeSeriesPredictor(
        target=AG_TS_TARGET,
        prediction_length=1,
        freq="h",
        known_covariates_names=AG_TS_KNOWN_COVARIATES,
        eval_metric="MAE",
    ).fit(
        train_data=ag_ts_dev_data,
        presets="best_quality",
        time_limit=3600,
    )

    # Keep the validation leaderboard and run settings in MLflow.
    ag_ts_leaderboard = autogluon_ts_predictor.leaderboard()
    mlflow.log_params({
        "autogluon_timeseries_version": version("autogluon.timeseries"),
        "prediction_length_hours": 1,
        "frequency": "hourly",
        "eval_metric": "MAE",
        "presets": "best_quality",
        "validation": "chronological_windows",
        "known_covariates": ",".join(AG_TS_KNOWN_COVARIATES),
        "development_hours": len(ag_ts_dev_data),
        "best_model": autogluon_ts_predictor.model_best,
    })
    mlflow.log_dict(
        ag_ts_leaderboard.to_dict(orient="records"),
        "development_leaderboard.json",
    )

display(ag_ts_leaderboard.head(10))
print(f"AutoGluon TimeSeries best model: {autogluon_ts_predictor.model_best}")
print(f"MLflow run ID: {autogluon_ts_run.info.run_id}")


**Note:**

- Missing-target hours stay in the hourly series as `NaN`; they are not dropped or filled with invented counts.
- AutoGluon uses previous-hour weather covariates and `cnt` history, not the manual rental-lag columns (`1h`, `24h`, `168h`).
- AutoGluon creates its own expanding chronological validation windows; it does not use LightGBM’s three folds.
- In this run, the best model was a WeightedEnsemble (96% Chronos2SmallFineTuned, 4% DynamicOptimizedTheta); DeepAR and TemporalFusionTransformer were skipped after a PyTorch matmul-precision error.


### Test Evaluation

Keep AutoGluon model selection within development. Generate one-hour-ahead forecasts across the test timeline with expanding history, so each forecast can use actual earlier test counts. Score only test hours with known rental counts, matching the LightGBM and TabPFN test hours.


In [ ]:
# Include development history and the held-out test timeline for rolling one-hour forecasts.
ag_ts_backtest_end = X_test.index.max()
ag_ts_test_period = hourly_data.loc[
    (hourly_data.index > X_dev.index.max())
    & (hourly_data.index <= ag_ts_backtest_end)
]
ag_ts_backtest_frame = hourly_data.loc[
    :ag_ts_backtest_end, [AG_TS_TARGET] + AG_TS_KNOWN_COVARIATES
].copy()
ag_ts_backtest_frame["weathersit_lag_1h"] = ag_ts_backtest_frame["weathersit_lag_1h"].astype("category")
ag_ts_backtest_frame["item_id"] = AG_TS_ITEM_ID
ag_ts_backtest_frame = ag_ts_backtest_frame.rename_axis("timestamp").reset_index()
ag_ts_backtest_data = TimeSeriesDataFrame.from_data_frame(
    ag_ts_backtest_frame,
    id_column="item_id",
    timestamp_column="timestamp",
)

# Each window forecasts one hour; later windows can use actual earlier test counts.
ag_ts_test_forecast_windows = autogluon_ts_predictor.backtest_predictions(
    data=ag_ts_backtest_data,
    num_val_windows=len(ag_ts_test_period),
    val_step_size=1,
)
ag_ts_all_test_forecasts = pd.concat(
    [window["mean"] for window in ag_ts_test_forecast_windows]
)
ag_ts_all_test_forecasts.index = ag_ts_all_test_forecasts.index.get_level_values("timestamp")
ag_ts_test_predictions = ag_ts_all_test_forecasts.reindex(y_test.index)

if ag_ts_test_predictions.isna().any():
    raise ValueError("AutoGluon did not return a forecast for every known-target test hour.")

# Score the same known-target test hours as LightGBM and TabPFN.
ag_ts_test_scores = regression_scores(y_test, ag_ts_test_predictions)
display(pd.DataFrame([ag_ts_test_scores], index=["AutoGluon Time Series"]).round(2))

with mlflow.start_run(run_id=autogluon_ts_run.info.run_id):
    mlflow.log_params({
        "test_strategy": "expanding_window_rolling_one_hour_ahead",
        "test_forecast_windows": len(ag_ts_test_period),
        "test_scored_hours": len(y_test),
    })
    mlflow.log_metrics({f"test_{name.lower()}": value for name, value in ag_ts_test_scores.items()})
    mlflow.log_dict(ag_ts_test_scores, "test_metrics.json")


**Note:**

- Start with the development history and selected AutoGluon model.
- Forecast one test hour ahead, add its actual count to the history when known, then forecast the next hour; keep the model fixed.
- Score known-count test hours with MAE and RMSE, then compare with LightGBM and TabPFN.


### Feature Importance

AutoGluon Time Series does not provide SHAP forecast explanations. Its built-in [permutation feature importance](https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.feature_importance.html) shuffles each weather covariate across timestamp rows in the supplied series, then measures the MAE change on the final one-hour forecast. Since only that hour is scored, treat the result as exploratory.


In [ ]:
# Use AutoGluon's built-in permutation importance for the selected model.
ag_ts_feature_importance = autogluon_ts_predictor.feature_importance(
    data=ag_ts_backtest_data,
    model=autogluon_ts_predictor.model_best,
    metric="MAE",
    features=AG_TS_KNOWN_COVARIATES,
    method="permutation",
    subsample_size=1,
    num_iterations=1,
    include_confidence_band=False,
    time_limit=120,
)

ag_ts_feature_importance = ag_ts_feature_importance.sort_values("importance")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ag_ts_feature_importance.index, ag_ts_feature_importance["importance"])
ax.axvline(0, color="gray", linewidth=1)
ax.set_title("AutoGluon Permutation Feature Importance")
ax.set_xlabel("Importance for MAE (higher = more important)")
fig.tight_layout()

with mlflow.start_run(run_id=autogluon_ts_run.info.run_id):
    mlflow.log_figure(fig, "interpretability/feature_importance.png")
    mlflow.log_dict(
        ag_ts_feature_importance.reset_index().to_dict(orient="records"),
        "interpretability/feature_importance.json",
    )
display(fig)
plt.close(fig)


#### Observations

- A positive score means shuffling a weather covariate increased MAE; a negative score means MAE decreased for this forecast.
- The plot ranks covariate importance, not the direction of each covariate’s effect. Values are shuffled across timestamp rows, but the score uses one final test hour, so treat the result as exploratory.


## Model Comparison

Compare LightGBM, TabPFN, and AutoGluon on the same test hours using MAE and RMSE. Lower values indicate smaller forecast errors.


In [ ]:
model_comparison = pd.DataFrame({
    "LightGBM": model_test,
    "TabPFN 3.5": tabpfn_scores,
    "AutoGluon Time Series": ag_ts_test_scores,
}).T
display(model_comparison.round(2))


### Observations

- TabPFN’s MAE is **23.57** rentals per hour, compared with **27.98** for AutoGluon Time Series and **29.40** for LightGBM. Their RMSE scores are **37.54**, **42.98**, and **47.93**, respectively.
- The previous-week baseline was less accurate on the hours where its lag was available; its results are shown in the LightGBM test evaluation.


**Note:** To view runs, start `mlflow server --port 5000` in a separate terminal from the same working directory, then open [http://localhost:5000](http://localhost:5000). Configure the tracking URI before the study if using a remote server.

## Summary

- **Dataset:** We use the UCI Bike Sharing hourly records from Washington, D.C.’s Capital Bikeshare system (**2011–2012**). Each row represents an hour; **`cnt`**, the total rentals in that hour, is the target, with calendar and weather information among the predictors.
- **Preparation:** The source has **165 missing hours**, so we add those timestamps temporarily to keep the **1-hour, 24-hour, and 168-hour rental lags** aligned with the clock. We also use **previous-hour weather**, convert columns to suitable types, and remove rows without a known rental count for tabular training and evaluation. AutoGluon keeps the complete development timeline and leaves missing counts as `NaN`.
- **Modeling:** The **latest 20% of known-count hours** is held out for testing. Three time-ordered development folds tune **LightGBM** with **Optuna**; LightGBM, **TabPFN**, and **AutoGluon Time Series** are evaluated on the same test hours. AutoGluon uses `best_quality`, its own chronological validation windows, `cnt` history, and previous-hour weather covariates. It selected a **WeightedEnsemble** of Chronos2SmallFineTuned (**96%**) and DynamicOptimizedTheta (**4%**); DeepAR and TemporalFusionTransformer were skipped after a PyTorch matmul-precision error.
- **Test results:** **MAE** is the average absolute error in rentals per hour; **RMSE** weighs large errors more heavily. TabPFN scored **23.57 MAE / 37.54 RMSE**, AutoGluon **27.98 / 42.98**, and LightGBM **29.40 / 47.93**. On the shared hours where the **previous-week baseline** is available, its MAE was **64.48**, versus **29.29** for LightGBM.
- **Forecast plots and explanations:** The forecast plot compares predictions with actual rentals over the **first test week**; residuals are actual minus predicted, so positive values indicate **underprediction** and negative values indicate **overprediction**. **SHAP** explains LightGBM and TabPFN predictions. AutoGluon’s built-in permutation importance shuffles weather covariate values across timestamp rows and measures the MAE change on the final one-hour forecast, so treat it as exploratory. These analyses describe model patterns, not causes.
